# Natural Gas Price Prediction using Qwen3 LLM

This notebook uses a Qwen3 8B model to analyze news headlines and summaries to predict natural gas price movements.

## 1. Install Dependencies

In [ ]:
!pip install requests

## 2. Import Libraries

In [ ]:
import requests
import json
from typing import Dict, List, Any

## 3. Configure API Endpoint

In [ ]:
API_ENDPOINT = "https://likely-flowing-shrimp.ngrok-free.app/v1/chat/completions"
MODEL_NAME = "hoangquan456/qwen3-nothink:8b"

## 4. Define Prediction Function

In [ ]:
def predict_price_change(news_headlines: List[str], news_summaries: List[str], yesterday_price: float) -> Dict[str, Any]:
    """
    Predict natural gas price change using Gemma LLM.
    
    Args:
        news_headlines: List of news headlines about energy
        news_summaries: List of corresponding news summaries
        yesterday_price: Yesterday's Henry Hub Natural Gas Spot Price
        
    Returns:
        Dictionary with 'trend' (up/down) and 'magnitude' (price change amount)
    """
    # Combine headlines and summaries into formatted text
    news_text = ""
    for i, (headline, summary) in enumerate(zip(news_headlines, news_summaries), 1):
        news_text += f"\nArticle {i}:\n"
        news_text += f"Headline: {headline}\n"
        news_text += f"Summary: {summary}\n"
    
    # Construct the prompt
    prompt = f"""Analyze the following news article headlines and summaries to determine the sentiment toward the natural gas market and predict whether the Henry Hub Natural Gas Spot Price will go up or down, and by how much. Yesterday's price is: ${yesterday_price:.2f}.

{news_text}

Output **only** a valid JSON object in the following format, with no explanations or text outside the JSON:

{{
"trend": "<up | down>",
"magnitude": <magnitude>
}}"""
    
    # Prepare API request
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "You are a financial analyst specialized in energy markets. You provide concise, data-driven predictions."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    
    # Make API request
    response = requests.post(
        API_ENDPOINT,
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=60
    )
    
    response.raise_for_status()
    
    # Parse response
    response_data = response.json()
    assistant_message = response_data['choices'][0]['message']['content']
    
    # Extract JSON from response (handle potential markdown formatting)
    assistant_message = assistant_message.strip()
    if assistant_message.startswith('```json'):
        assistant_message = assistant_message[7:]
    if assistant_message.startswith('```'):
        assistant_message = assistant_message[3:]
    if assistant_message.endswith('```'):
        assistant_message = assistant_message[:-3]
    assistant_message = assistant_message.strip()
    
    # Parse JSON prediction
    prediction = json.loads(assistant_message)
    
    return prediction

## 5. Hardcoded Test Data

In [ ]:
# Yesterday's natural gas price
yesterday_price = 2.50

# Example news headlines
news_headlines = [
    "Natural Gas Inventories Fall Short of Expectations",
    "Severe Winter Storm to Hit Northeast, Heating Demand Expected to Surge",
    "Major LNG Export Facility Announces Unexpected Maintenance Shutdown",
    "Renewable Energy Growth Slows as Natural Gas Remains Competitive",
    "Analysts Predict Tight Natural Gas Market Conditions Through Winter"
]

# Corresponding news summaries
news_summaries = [
    "Weekly natural gas inventory data shows a smaller-than-expected build, with storage levels falling below the five-year average. Market participants express concerns about supply adequacy heading into peak winter demand season.",
    "The National Weather Service has issued warnings for a major winter storm system expected to impact the Northeast and Midwest regions. Forecasters predict temperatures to drop significantly below seasonal averages, driving increased heating demand across residential and commercial sectors.",
    "One of the largest LNG export terminals in the Gulf Coast region has announced an unscheduled shutdown for critical maintenance work. The facility, which handles significant volumes of natural gas exports, is expected to be offline for at least two weeks, potentially tightening domestic supply.",
    "Despite ambitious renewable energy targets, natural gas continues to maintain its competitive position in the energy mix. Recent data shows a slowdown in wind and solar capacity additions, with utilities citing favorable natural gas economics and reliability concerns.",
    "Leading energy analysts are forecasting tight market conditions for natural gas through the remainder of the winter season. Factors including robust export demand, weather uncertainties, and constrained production growth are expected to support elevated price levels."
]

## 6. Run Prediction

In [ ]:
print("="*80)
print("NATURAL GAS PRICE PREDICTION - GEMMA LLM")
print("="*80)
print(f"\nYesterday's Price: ${yesterday_price:.2f}")
print(f"\nNumber of News Articles: {len(news_headlines)}")
print("\nNews Headlines:")
for i, headline in enumerate(news_headlines, 1):
    print(f"  {i}. {headline}")

print("\n" + "="*80)
print("Making prediction...")
print("="*80 + "\n")

try:
    prediction = predict_price_change(news_headlines, news_summaries, yesterday_price)
    
    print("PREDICTION RESULT:")
    print(f"  Trend: {prediction['trend'].upper()}")
    print(f"  Magnitude: ${prediction['magnitude']:.2f}")
    
    # Calculate predicted price
    if prediction['trend'].lower() == 'up':
        predicted_price = yesterday_price + prediction['magnitude']
        print(f"\nPredicted Price: ${predicted_price:.2f} (${prediction['magnitude']:+.2f})")
    else:
        predicted_price = yesterday_price - prediction['magnitude']
        print(f"\nPredicted Price: ${predicted_price:.2f} (${-prediction['magnitude']:+.2f})")
    
    print("\n" + "="*80)
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 7. Test with Different News Scenarios

In [ ]:
# Test Case 1: Bearish news (price should go down)
bearish_headlines = [
    "Mild Winter Forecast Reduces Heating Demand Outlook",
    "New Pipeline Capacity to Ease Supply Constraints",
    "Natural Gas Production Reaches Record High in Permian Basin"
]

bearish_summaries = [
    "Meteorologists are predicting warmer-than-average temperatures for the upcoming winter season, which could significantly reduce heating demand for natural gas across major consumption regions.",
    "A major new pipeline project has been completed ahead of schedule, adding substantial transportation capacity that will help alleviate regional supply bottlenecks and improve gas flow to key markets.",
    "Natural gas production in the Permian Basin has reached an all-time high, with producers reporting robust output growth driven by improved drilling efficiency and favorable well economics."
]

print("\nTest Case 1: Bearish News Scenario")
print("="*80)
try:
    prediction = predict_price_change(bearish_headlines, bearish_summaries, yesterday_price)
    print(f"Prediction: {prediction}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Test Case 2: Bullish news (price should go up)
bullish_headlines = [
    "Polar Vortex Expected to Drive Record Natural Gas Demand",
    "Major Production Facility Shut Down Due to Equipment Failure",
    "Asian LNG Buyers Increase US Natural Gas Import Orders"
]

bullish_summaries = [
    "A severe polar vortex event is forecast to bring extreme cold temperatures across major population centers, with meteorologists predicting potential record-breaking natural gas demand for heating as temperatures plunge well below normal levels.",
    "One of the largest natural gas production facilities in the Gulf region has experienced a critical equipment failure, forcing an immediate shutdown. The facility accounts for approximately 3% of total US natural gas production.",
    "Multiple Asian LNG importing nations have significantly increased their purchase orders for US natural gas exports, citing strong demand and tight global LNG markets. This surge in export demand is expected to draw down domestic supply."
]

print("\nTest Case 2: Bullish News Scenario")
print("="*80)
try:
    prediction = predict_price_change(bullish_headlines, bullish_summaries, yesterday_price)
    print(f"Prediction: {prediction}")
except Exception as e:
    print(f"Error: {e}")

## 8. Notes

### Model Configuration:
- Using Qwen3 8B quantized model (qat)
- Endpoint hosted on ngrok tunnel
- System prompt configures model as financial analyst

### Output Format:
- `trend`: "up" or "down"
- `magnitude`: Dollar amount of predicted price change

### Future Improvements:
1. Load actual scraped news from JSONL files
2. Integrate with historical price data (DHHNGSP.csv)
3. Batch process multiple days
4. Compare predictions with actual price movements
5. Track model accuracy over time
6. Add error handling for API timeouts/failures
7. Implement retry logic for failed requests